In [ ]:
!pip install pillow-heif
!pip install torch_optimizer
!pip install -U albumentations

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
import numpy as np
import gc
import os
from PIL import Image
import torch.optim as optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import pillow_heif

# Register HEIC
pillow_heif.register_heif_opener()

PATCH_SIZE = 128
def keep_first_channel(img, **kwargs):
    return img[..., :1]

albumentations_transform = A.Compose([
    A.Resize(height=256, width=300),
    A.CenterCrop(height=PATCH_SIZE, width=PATCH_SIZE),
    A.ToGray(p=1.0),
    A.Lambda(image=keep_first_channel, p=1.0),  # Keep only the first channel
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.96, 1.04), translate_percent=0.01, rotate=0, p=0.25),
    A.HorizontalFlip(p=0.5),
    # A.VerticalFlip(p=0.5),
    # A.RandomRotate90(p=0.5),
    # A.Transpose(p=0.5),
    A.Affine(scale=(0.96,1.04), translate_percent=0.01, rotate=0, p=0.25),

    A.RandomBrightnessContrast(p=0.5),
    A.RandomGamma(p=0.25),
    A.Emboss(p=0.25),
    A.Blur(blur_limit=3, p=0.01),
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=6, p=1.0),
        A.GridDistortion(p=1.0),
        A.OpticalDistortion(distort_limit=2.0, p=1.0),
    ], p=0.8),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])



In [ ]:
# Data Preprocessing Pipeline
# 1. Custom Dataset Class for handling various image formats including HEIC

class CustomImageDataset(Dataset):
    """
    Custom dataset class for loading face images with support for multiple formats.
    Handles HEIC files using pillow-heif library and applies augmentations.
    """
    def __init__(self, image_dir):
        self.image_dir = image_dir
        # Support for common image formats including HEIC
        self.image_files = [f for f in os.listdir(image_dir) 
                           if f.lower().endswith(('.jpg', '.jpeg', '.png', '.heic'))]
        print(f"Found {len(self.image_files)} images in dataset")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_files[idx])
        # Convert to RGB to ensure 3-channel input for preprocessing
        img = Image.open(path).convert('RGB')
        img_np = np.array(img)  # Convert to numpy for Albumentations
        return img_np, self.image_files[idx]

In [ ]:
def convert_heic_to_jpeg(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all HEIC files from input folder
    heic_files = [f for f in os.listdir(input_folder) 
                  if f.lower().endswith('.heic')]
    
    print(f"{len(heic_files)} HEIC files to convert")
    
    converted_count = 0
    for heic_file in heic_files:
        try:
            # Open HEIC image
            heic_path = os.path.join(input_folder, heic_file)
            image = Image.open(heic_path)
            
            # Convert to RGB if needed
            if image.mode != 'RGB':
                image = image.convert('RGB')
            
            # Create output filename with .jpg extension
            jpeg_filename = os.path.splitext(heic_file)[0] + '.jpg'
            jpeg_path = os.path.join(output_folder, jpeg_filename)
            
            # Save as JPEG
            image.save(jpeg_path, 'JPEG', quality=95)
            
            converted_count += 1
            print(f"Converted: {heic_file} -> {jpeg_filename}")
            
        except Exception as e:
            print(f"Error converting {heic_file}: {str(e)}")
    
    print(f"\nDone: {converted_count}/{len(heic_files)} files converted successfully")

In [ ]:
# 2. Data Augmentation Pipeline using Albumentations
# Comprehensive augmentation strategy to expand limited dataset

def keep_first_channel(img, **kwargs):
    """Helper function to ensure single-channel output after grayscale conversion"""
    return img[..., :1]

PATCH_SIZE = 128  # Target image size for GAN training

# Augmentation pipeline with geometric and photometric transformations
albumentations_transform = A.Compose([
    # Geometric preprocessing
    A.Resize(height=256, width=300),  # Standardize input size
    A.CenterCrop(height=PATCH_SIZE, width=PATCH_SIZE),  # Focus on central features
    
    # Color space conversion
    A.ToGray(p=1.0),  # Convert to grayscale
    A.Lambda(image=keep_first_channel, p=1.0),  # Ensure single channel
    
    # Geometric augmentations
    A.HorizontalFlip(p=0.5),  # Mirror faces for variety
    A.Affine(scale=(0.96, 1.04), translate_percent=0.01, rotate=0, p=0.25),
    
    # Photometric augmentations
    A.RandomBrightnessContrast(p=0.5),  # Lighting variations
    A.RandomGamma(p=0.25),  # Gamma correction
    A.Emboss(p=0.25),  # Texture enhancement
    A.Blur(blur_limit=3, p=0.01),  # Minimal blur for realism
    
    # Advanced deformations for robustness
    A.OneOf([
        A.ElasticTransform(alpha=120, sigma=6, p=1.0),
        A.GridDistortion(p=1.0),
        A.OpticalDistortion(distort_limit=2.0, p=1.0),
    ], p=0.8),
    
    # Normalization for GAN training ([-1, 1] range)
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),  # Convert to PyTorch tensor
])

In [ ]:
# 3. Dataset Expansion Process
# Generate multiple augmented versions of each original image

def expand_dataset(original_dataset, augmentation_transform, save_dir, versions_per_image=12):
    """
    Expand dataset by creating multiple augmented versions of each image.
    
    Args:
        original_dataset: CustomImageDataset instance
        augmentation_transform: Albumentations compose transform
        save_dir: Directory to save augmented images
        versions_per_image: Number of augmented versions per original image
    """
    os.makedirs(save_dir, exist_ok=True)
    train_loader = DataLoader(original_dataset, batch_size=1, shuffle=False)
    
    total_generated = 0
    for img_np_batch, filenames in train_loader:
        img_np = img_np_batch[0].numpy()  # Remove batch dimension
        original_name = os.path.splitext(filenames[0])[0]

        # Generate multiple augmented versions
        for version in range(1, versions_per_image + 1):
            augmented = augmentation_transform(image=img_np)
            img_aug = augmented["image"]

            # Unnormalize for saving: [-1,1] → [0,1] → [0,255]
            img_aug = img_aug * 0.5 + 0.5
            img_aug = img_aug.numpy().transpose(1, 2, 0)  # C,H,W → H,W,C
            img_aug = (img_aug * 255).clip(0, 255).astype(np.uint8)

            if img_aug.shape[2] == 1:
                img_aug = img_aug.squeeze(-1)  # Remove channel dimension for grayscale

            save_path = os.path.join(save_dir, f"{original_name}_aug_{version:02}.jpg")
            cv2.imwrite(save_path, img_aug)
            total_generated += 1

        print(f"Generated {versions_per_image} versions for: {original_name}")
    
    print(f"Dataset expansion complete: {total_generated} total images generated")
    return total_generated

In [ ]:
def show_images(images, nrow=8, title=None):
    grid = make_grid(images, nrow=nrow)
    plt.figure(figsize=(10, 10))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()  
def save_images(images, filename):
    grid = make_grid(images, nrow=8, normalize=True, scale_each=True)
    save_image(grid, filename)
    print(f"Images saved to {filename}")

def plot_images_from_loader(loader, num_images=64, save=False):
    images = []
    for i, (img_batch, _) in enumerate(loader):
        if i * loader.batch_size >= num_images:
            break
        images.append(img_batch)
    
    images = torch.cat(images, dim=0)[:num_images]
    
    if save:
        save_images(images, "sample_images.png")
    else:
        show_images(images, nrow=8)

plot_images_from_loader(train_loader, num_images=64, save=True)

In [ ]:
# # class Generator(nn.Module):
# #     def __init__(self, noise_dim):
# #         super(Generator, self).__init__()
# #         self.model = nn.Sequential(
# #             nn.Linear(noise_dim, 1024*32*32),
# #             nn.ReLU(True),
# #             nn.Unflatten(1, (1024, 32, 32)),
# #             nn.ConvTranspose2d(1024, 512, kernel_size=4, stride=2, padding=1, output_padding=0),
# #             nn.ReLU(True),
# #             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, output_padding=0),
# #             nn.ReLU(True),
# #             nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, output_padding=0),
# #             nn.ReLU(True),
# #             nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1, output_padding=0),
# #             nn.Tanh()
# #         )

# #     def forward(self, x):
# #         return self.model(x)



class Generator(nn.Module):
    """
    Generator network for face image generation.
    
    Architecture:
    - Input: 100-dimensional noise vector
    - Fully connected layer with batch normalization
    - Progressive upsampling from 8x8 to 128x128
    - Bilinear upsampling with convolutional refinement
    - Output: 128x128 single-channel face image
    """
    
    def __init__(self, noise_dim=100):
        super(Generator, self).__init__()
        self.noise_dim = noise_dim
        
        # Initial fully connected layer to create base feature maps
        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 256 * 8 * 8),  # Map to 256 8x8 feature maps
            nn.BatchNorm1d(256 * 8 * 8),        # Normalize for stable training
            nn.ReLU(True)                       # Non-linearity
        )
        
        # Progressive upsampling blocks with feature refinement
        self.up_blocks = nn.Sequential(
            # 8x8 → 16x16 (256 → 128 channels)
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # 16x16 → 32x32 (128 → 64 channels)
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # 32x32 → 64x64 (64 → 32 channels)
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            
            # 64x64 → 128x128 (32 → 1 channel)
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Tanh()  # Output in [-1, 1] range matching normalized data
        )

    def forward(self, x):
        """
        Forward pass through generator.
        
        Args:
            x: Random noise tensor of shape (batch_size, noise_dim)
        
        Returns:
            Generated images of shape (batch_size, 1, 128, 128)
        """
        # Transform noise to initial feature maps
        x = self.fc(x)                    # (B, 256*8*8)
        x = x.view(-1, 256, 8, 8)         # Reshape to (B, 256, 8, 8)
        
        # Progressive upsampling and refinement
        x = self.up_blocks(x)             # (B, 1, 128, 128)
        return x

# Test generator architecture
def test_generator():
    """Test generator with sample input"""
    gen = Generator(noise_dim=100)
    test_noise = torch.randn(4, 100)  # Batch of 4 noise vectors
    output = gen(test_noise)
    print(f"Generator output shape: {output.shape}")  # Should be (4, 1, 128, 128)
    print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")
    return gen

# Uncomment to test
# test_generator()

In [ ]:
# class Generator(nn.Module):
#     def __init__(self, noise_dim=100):
#         super(Generator, self).__init__()
#         self.fc = nn.Sequential(
#             nn.Linear(noise_dim, 128 * 8 * 8),
#             nn.ReLU(True)
#         )

#         self.net = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear'),  # 16x16
#             nn.Conv2d(128, 64, kernel_size=3, padding=1),
#             nn.ReLU(True),

#             nn.Upsample(scale_factor=2, mode='bilinear'),  # 32x32
#             nn.Conv2d(64, 32, kernel_size=3, padding=1),
#             nn.ReLU(True),

#             nn.Upsample(scale_factor=2, mode='bilinear'),  # 64x64
#             nn.Conv2d(32, 16, kernel_size=3, padding=1),
#             nn.ReLU(True),

#             nn.Upsample(scale_factor=2, mode='bilinear'),  # 128x128
#             nn.Conv2d(16, 1, kernel_size=3, padding=1),
#             nn.Tanh()
#         )

#     def forward(self, x):
#         x = self.fc(x)                  # (B, 128*8*8)
#         x = x.view(-1, 128, 8, 8)       # (B, 128, 8, 8)
#         return self.net(x)             # (B, 1, 128, 128)


In [ ]:
class Discriminator(nn.Module):
    """
    Discriminator network for real/fake image classification.
    
    Architecture:
    - Input: 128x128 single-channel face image
    - Progressive downsampling through convolutional layers
    - LeakyReLU activations to prevent vanishing gradients
    - No batch normalization to maintain gradient flow
    - Output: Single probability score for real/fake classification
    """
    
    def __init__(self):
        super(Discriminator, self).__init__()

        self.net = nn.Sequential(
            # 128x128 → 64x64 (1 → 16 channels)
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),  # Negative slope prevents dying neurons

            # 64x64 → 32x32 (16 → 32 channels)
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32 → 16x16 (32 → 64 channels)
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # 16x16 → 8x8 (64 → 128 channels)
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # 8x8 → 4x4 (128 → 256 channels)
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # Flatten and classify
            nn.Flatten(),                           # (B, 256*4*4)
            nn.Linear(256 * 4 * 4, 1),             # (B, 1)
            nn.Sigmoid()                           # Probability output [0, 1]
        )

    def forward(self, x):
        """
        Forward pass through discriminator.
        
        Args:
            x: Input images of shape (batch_size, 1, 128, 128)
        
        Returns:
            Probability scores of shape (batch_size, 1) indicating real/fake
        """
        return self.net(x)

# Test discriminator architecture
def test_discriminator():
    """Test discriminator with sample input"""
    disc = Discriminator()
    test_images = torch.randn(4, 1, 128, 128)  # Batch of 4 images
    output = disc(test_images)
    print(f"Discriminator output shape: {output.shape}")  # Should be (4, 1)
    print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")
    return disc

# Uncomment to test
# test_discriminator()

In [ ]:
# Using available hardware device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Creating the generator and discriminator
latent_dim = 100
generator = Generator(latent_dim).to(device)
discriminator = Discriminator().to(device)

# creating binary cross entropy loss instancea
criterion = nn.BCELoss()


In [ ]:
device

In [ ]:
from torch_optimizer import RAdam, Lookahead


base_gen_optim = RAdam(generator.parameters(), lr=0.00017558485508404896, betas=(0.5, 0.999))
generator_optimizer = Lookahead(base_gen_optim)

base_disc_optim = RAdam(discriminator.parameters(), lr=0.00017558485508404896, betas=(0.5, 0.999))
discriminator_optimizer = Lookahead(base_disc_optim)


# generator_optimizer = RAdam(generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
# discriminator_optimizer = RAdam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))


# Checkpoint directory and paths
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
generator_checkpoint_path = os.path.join(checkpoint_dir, "generator.pth")
discriminator_checkpoint_path = os.path.join(checkpoint_dir, "discriminator.pth")

start_epoch = 0

def save_checkpoint(model, optimizer, path, epoch):
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch
    }
    torch.save(checkpoint, path)

def load_checkpoint(model, optimizer, path):
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        epoch = checkpoint.get('epoch', 0)
        print(f"Checkpoint loaded from {path}, epoch {epoch}")
        return epoch
    else:
        print(f"No checkpoint found at {path}")
        return 0

# Load checkpoints if available
if os.path.exists(generator_checkpoint_path):
    start_epoch = load_checkpoint(generator, generator_optimizer, generator_checkpoint_path)
else:
    print("No generator checkpoint found, starting from scratch.")

if os.path.exists(discriminator_checkpoint_path):
    _ = load_checkpoint(discriminator, discriminator_optimizer, discriminator_checkpoint_path)
else:
    print("No discriminator checkpoint found, starting from scratch.")

In [ ]:
num_epochs = 200
gen_losses = []
disc_losses = []

In [ ]:
best_g_loss = float('inf')  # Track best generator loss

for epoch in range(start_epoch, num_epochs):
    d_epoch_loss = 0.0
    g_epoch_loss = 0.0

    for i, (real_images, _) in enumerate(train_loader):
        batch_size = real_images.size(0)
        real_images = real_images.to(device)

        # Label smoothing: real = 0.9, fake = 0.1
        real_labels = torch.full((batch_size, 1), 0.9, device=device)
        fake_labels = torch.full((batch_size, 1), 0.1, device=device)

        ### -------- Train Discriminator --------
        discriminator_optimizer.zero_grad()

        # Real
        real_outputs = discriminator(real_images)
        d_real_loss = criterion(real_outputs, real_labels)

        # Fake
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(z)
        fake_outputs = discriminator(fake_images.detach())
        d_fake_loss = criterion(fake_outputs, fake_labels)

        # Total Discriminator Loss
        d_loss = d_real_loss + d_fake_loss
        d_epoch_loss += d_loss.item()

        d_loss.backward()
        discriminator_optimizer.step()

        ### -------- Train Generator --------
        generator_optimizer.zero_grad()

        outputs = discriminator(fake_images)
        g_loss = criterion(outputs, real_labels)  # try to fool discriminator
        g_epoch_loss += g_loss.item()

        g_loss.backward()
        generator_optimizer.step()

        if (i + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], "
                  f"D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}, "
                  f"D(x): {real_outputs.mean().item():.2f}, D(G(z)): {fake_outputs.mean().item():.2f}")

    # Average losses
    avg_g_loss = g_epoch_loss / len(train_loader)
    avg_d_loss = d_epoch_loss / len(train_loader)
    gen_losses.append(avg_g_loss)
    disc_losses.append(avg_d_loss)

    ### -------- Save best model every 5 epochs --------
    if (epoch + 1) % 10 == 0:
        # if avg_g_loss < best_g_loss:
        best_g_loss = avg_g_loss
        torch.save(generator.state_dict(), os.path.join(checkpoint_dir, f"best_generator_epoch{epoch+1}.pth"))
        torch.save(discriminator.state_dict(), os.path.join(checkpoint_dir, f"best_discriminator_epoch{epoch+1}.pth"))
        print(f"Best model saved at epoch {epoch+1} (G Loss: {best_g_loss:.4f})")

    ### -------- Visualization every 10 epochs --------
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            z_vis = torch.randn(16, latent_dim).to(device)
            gen_imgs = generator(z_vis).cpu()
            grid = make_grid(gen_imgs, nrow=4, normalize=True, scale_each=True)
            plt.figure(figsize=(12, 12))
            plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
            plt.axis('off')
            plt.title(f"Epoch {epoch+1} Generated Images")
            plt.show()

In [ ]:
generator.eval()

with torch.no_grad():
    noise = torch.randn(64, latent_dim).to(device)
    fake_images = generator(noise)
    fake_images = (fake_images + 1) / 2  # Rescale to [0, 1]
    grid = make_grid(fake_images, nrow=8, normalize=True)
    save_image(grid, "generated_images_final.png")

# Display the generated images
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
ax.axis('off')
plt.show()


In [ ]:
#Display the losses
plt.figure(figsize=(10, 5))
plt.plot(gen_losses, label='Generator Loss', color='blue')
plt.plot(disc_losses, label='Discriminator Loss', color='red')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Generator and Discriminator Losses')
plt.legend()
plt.show()

In [ ]:
# Save the final models
torch.save(generator.state_dict(), generator_checkpoint_path)
torch.save(discriminator.state_dict(), discriminator_checkpoint_path)

In [ ]:
# save losses to a file
import json
losses = {
    'generator_losses': gen_losses,
    'discriminator_losses': disc_losses
}
with open('losses.json', 'w') as f:
    json.dump(losses, f)

In [ ]:
checkpoint_dir = "checkpoints"
generator_checkpoint_path = os.path.join(checkpoint_dir, "generator.pth")
discriminator_checkpoint_path = os.path.join(checkpoint_dir, "discriminator.pth")

In [ ]:
# Generate a random image using random latent vector and compare similarity with the original image
def generate_random_image(generator, latent_dim=100):
    noise = torch.randn(1, latent_dim).to(device)
    with torch.no_grad():
        generated_image = generator(noise)
    return generated_image

# Function to calculate similarity between two images
def calculate_similarity(image1, image2):
    # Flatten the images and calculate cosine similarity
    image1_flat = image1.view(-1)
    image2_flat = image2.view(-1)
    similarity = torch.nn.functional.cosine_similarity(image1_flat, image2_flat, dim=0)
    return similarity.item()

# Example usage
original_image, _ = next(iter(train_loader))
original_image = original_image[0].to(device)  # Take the first image from the batch
generated_image = generate_random_image(generator)  
similarity = calculate_similarity(original_image, generated_image.squeeze())
print(f"Similarity between original and generated image: {similarity:.4f}")



In [ ]:
# Model Architecture Summary and Comparison
def print_model_summary():
    """Print summary of both generator and discriminator architectures"""
    
    print("="*60)
    print("GAN ARCHITECTURE SUMMARY")
    print("="*60)
    
    # Generator summary
    generator = Generator(noise_dim=100)
    total_params_gen = sum(p.numel() for p in generator.parameters())
    trainable_params_gen = sum(p.numel() for p in generator.parameters() if p.requires_grad)
    
    print(f"\nGENERATOR:")
    print(f"  Input: Random noise vector (100 dimensions)")
    print(f"  Output: Face image (1 × 128 × 128)")
    print(f"  Total parameters: {total_params_gen:,}")
    print(f"  Trainable parameters: {trainable_params_gen:,}")
    print(f"  Architecture: FC → Upsample × 4 → Conv2D")
    
    # Discriminator summary  
    discriminator = Discriminator()
    total_params_disc = sum(p.numel() for p in discriminator.parameters())
    trainable_params_disc = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
    
    print(f"\nDISCRIMINATOR:")
    print(f"  Input: Face image (1 × 128 × 128)")
    print(f"  Output: Probability score (1 dimension)")
    print(f"  Total parameters: {total_params_disc:,}")
    print(f"  Trainable parameters: {trainable_params_disc:,}")
    print(f"  Architecture: Conv2D × 5 → FC → Sigmoid")
    
    print(f"\nTOTAL NETWORK PARAMETERS: {total_params_gen + total_params_disc:,}")
    print("="*60)

# Print the model summary
print_model_summary()